# Bank Branch Expansion Simulator
## Exploratory Data Analysis — FDIC Summary of Deposits

This notebook walks through the full EDA pipeline for the project:

1. Load and inspect the raw SOD data
2. Clean and validate
3. Analyse deposit distributions
4. Explore branch density by state and ZIP
5. Compute the Opportunity Score and Expected Capture
6. Identify top expansion targets
7. Export processed CSVs for Tableau


In [24]:
# ── Imports ──────────────────────────────────────────────────────────────── 

import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt 
import plotly.express as px 
import plotly.graph_objects as go 
from pathlib import Path 

pd.set_option('display.float_format','{:,.2f}'.format)
pd.set_option('display.max_columns',30)

In [26]:
# ── Configuration ──────────────────────────────────────────────────────────

Root = Path(r'C:\Users\chand\Desktop\Data Science\ORU Course\Bank Branch Expansion Analysis\Dataset')
Data_Path = Root/'FDIC_SOD_2025.csv'
Output_Dir = Root/'Data'/'Processed'
Output_Dir.mkdir(parents = True, exist_ok = True)

print(f"Data path:{Data_Path}")
print(f"Output dir:{Output_Dir}")

Data path:C:\Users\chand\Desktop\Data Science\ORU Course\Bank Branch Expansion Analysis\Dataset\FDIC_SOD_2025.csv
Output dir:C:\Users\chand\Desktop\Data Science\ORU Course\Bank Branch Expansion Analysis\Dataset\Data\Processed


---
## 1. Load and Inspect Raw Data

In [16]:
df = pd.read_csv(Data_Path)
print(f'Shape:{df.shape}')
df.head(5)

Shape:(76120, 12)


,YEAR,CERT,NAMEFULL,ADDRESBR,BRNUM,UNINUMBR,STALPBR,STALP,DEPSUMBR,SIMS_LATITUDE,SIMS_LONGITUDE,ZIPBR
0,2025,14,State Street Bank and Trust Company,"One Congress Street, Suite 1",0,6,MA,MA,207849000,42.36,-71.06,2114
1,2025,14,State Street Bank and Trust Company,"2323 Grand Boulevard, Floor 5",33,667916,MO,MA,0,39.09,-94.58,64108
2,2025,14,State Street Bank and Trust Company,"111 Town Square Place,suite 735",34,671182,NJ,MA,0,40.73,-74.03,7310
3,2025,35,AuburnBank,100 N Gay St,0,17,AL,AL,553080,32.61,-85.48,36830
4,2025,35,AuburnBank,1851 Frederick Rd,10,478897,AL,AL,60269,32.62,-85.40,36801


In [20]:
df = df.rename(columns = {
    'ADDRESBR':'ADDRESS',
    'NAMEFULL':'BANK_NAME',
    'ZIPBR':'ZIP',
    'STALPBR':'STATE',
    'SIMS_LONGITUDE':'LON',
    'SIMS_LATITUDE':'LAT',
    'DEPSUMBR':'DEPOSITS_USD'
})

print(f'Shape:{df.shape}')
df.head(5)

Shape:(76120, 12)


,YEAR,CERT,BANK_NAME,ADDRESS,BRNUM,UNINUMBR,STATE,STALP,DEPOSITS_USD,LAT,LON,ZIP
0,2025,14,State Street Bank and Trust Company,"One Congress Street, Suite 1",0,6,MA,MA,207849000,42.36,-71.06,2114
1,2025,14,State Street Bank and Trust Company,"2323 Grand Boulevard, Floor 5",33,667916,MO,MA,0,39.09,-94.58,64108
2,2025,14,State Street Bank and Trust Company,"111 Town Square Place,suite 735",34,671182,NJ,MA,0,40.73,-74.03,7310
3,2025,35,AuburnBank,100 N Gay St,0,17,AL,AL,553080,32.61,-85.48,36830
4,2025,35,AuburnBank,1851 Frederick Rd,10,478897,AL,AL,60269,32.62,-85.40,36801


In [ ]:
# Missing Values Summary

missing = df.isnull().sum()
missing_pct = (missing/len(df) *100).round(2)
missing_df = pd.DataFrame({'Missing Count':missing, 'Missing %':missing_pct})
missing_df[missing_df['Missing Count']>0]

,Missing Count,Missing %


In [29]:
# Fixing the Data ----------

# 1. Remove branches with 0 deposits

df = df[df['DEPOSITS_USD']>0].copy()

# 2. Insuring that ZIP codes are 5 digits strings (pad with 0's if needed)

df['ZIP'] = df['ZIP'].astype(str).str.zfill(5)

# 3. Optional: Remove rows with missing lat/lon (if needed for mapping)

df = df.dropna(subset=['LAT','LON'])

# 4. Optional: Reset the Index

df.reset_index(drop=True,inplace=True)

# 5. Change Deposits to Actual $$ (from 1000's)

df['DEPOSITS_USD'] = df['DEPOSITS_USD']*1000

print(f"After Cleaning: {len(df):,} branches remain")

After Cleaning: 73,876 branches remain


In [28]:
df.head(10)

,YEAR,CERT,BANK_NAME,ADDRESS,BRNUM,UNINUMBR,STATE,STALP,DEPOSITS_USD,LAT,LON,ZIP
0,2025,14,State Street Bank and Trust Company,"One Congress Street, Suite 1",0,6,MA,MA,207849000,42.36,-71.06,02114
1,2025,35,AuburnBank,100 N Gay St,0,17,AL,AL,553080,32.61,-85.48,36830
2,2025,35,AuburnBank,1851 Frederick Rd,10,478897,AL,AL,60269,32.62,-85.40,36801
3,2025,35,AuburnBank,2315 Bent Creek Rd,12,493224,AL,AL,94392,32.61,-85.43,36830
4,2025,35,AuburnBank,132 Fob James Dr,13,531906,AL,AL,54872,32.82,-85.18,36854
5,2025,35,AuburnBank,215 S 6th St,3,180886,AL,AL,107955,32.65,-85.38,36801
6,2025,35,AuburnBank,1351 S Donahue Dr,7,361391,AL,AL,52561,32.58,-85.49,36832
7,2025,35,AuburnBank,950 Auburn Rd,8,364622,AL,AL,17566,32.57,-85.66,36866
8,2025,39,Robertson Banking Company,216 N Walnut Ave,0,21,AL,AL,233726,32.52,-87.84,36732
9,2025,39,Robertson Banking Company,1406 Us Highway 80 W,2,180895,AL,AL,32681,32.50,-87.85,36732


---
## 2. Summary Statistics

In [31]:
print("=== Dataset Summary ===")
print(f'Total Branches : {len(df):,}')
print(f'Unique Branches : {df['BANK_NAME'].nunique():,}')
print(f'Unique ZIPs : {df['ZIP'].nunique():,}')
print(f'States Covered : {df['STATE'].nunique():,}')
print(f'Total Deposits: ${df['DEPOSITS_USD'].sum():,.0f}')
print(f'Average Deposit Per Branch: ${df['DEPOSITS_USD'].mean():,.0f}')
print(f'Median Deposits: ${df['DEPOSITS_USD'].median():,.0f}')

=== Dataset Summary ===
Total Branches : 73,876
Unique Branches : 3,895
Unique ZIPs : 17,971
States Covered : 58
Total Deposits: $18,103,095,758,000
Average Deposit Per Branch: $245,047,049
Median Deposits: $82,054,000


---
## 3. Deposit Distribution Analysis

In [36]:
# Deposit per branch — exclude top 1% to remove outlier skew

dep_m = df['DEPOSITS_USD']/1e6
p99 = dep_m.quantile(0.99)

fig = px.histogram(
    dep_m[dep_m<p99],
    nbins = 100,
    title = 'Deposits Per Branch (up to 99th percentile)',
    labels= {'value':'Deposits ($M)'},
    color_discrete_sequence=['#2563EB']
)
fig.update_layout(bargap = 0.05, showlegend = False)
fig.show()

print(f'Branches < $1M deposits : {(dep_m < 1).sum():,} ({(dep_m < 1).mean()*100:.1f}%)')
print(f'Branches $1M - $100M : {((dep_m > 1) & (dep_m < 100)).sum():,}')
print(f'Brances > $100M deposits : {(dep_m >= 100).sum():,}')

Branches < $1M deposits : 400 (0.5%)
Branches $1M - $100M : 42,650
Brances > $100M deposits : 30,821


---
## 4. State-Level Analysis

In [43]:
# Group Data at State Level

state_df = df.groupby('STATE',as_index=False).agg(
    TOTAL_DEPOSITS = ('DEPOSITS_USD','sum'),
    BRANCH_COUNT = ('CERT','count'),
    ZIP_COUNT = ('ZIP','nunique') 
    )

state_df['AVG_DEPOSIT_PER_BRANCH'] = state_df['TOTAL_DEPOSITS']/state_df['BRANCH_COUNT']
state_df = state_df.sort_values('TOTAL_DEPOSITS',ascending=False)

state_df.head(5)


,STATE,TOTAL_DEPOSITS,BRANCH_COUNT,ZIP_COUNT,AVG_DEPOSIT_PER_BRANCH
38,NY,2613706304000,3935,910,"664,220,153.49"
4,CA,1796631056000,5425,1058,"331,176,231.52"
49,TX,1498234125000,6000,1222,"249,705,687.50"
50,UT,1069658813000,499,124,"2,143,604,835.67"
47,SD,919583000000,421,188,"2,184,282,660.33"


In [47]:
fig = px.bar(
    state_df.head(20),
    x = 'TOTAL_DEPOSITS', y = 'STATE',
    orientation='h',
    title = 'Top 20 States by Total Deposits',
    labels = {'TOTAL_DEPOSITS': 'Total desposits ($)', 'STATE':'State'},
    color='TOTAL_DEPOSITS',color_continuous_scale='Blues'
)

fig.update_layout(yaxis = dict(autorange = 'reversed'), coloraxis_showscale = False, showlegend = False)
fig.show()

In [48]:
# Branch count by state

fig = px.bar(
    state_df.head(20),
    x='BRANCH_COUNT', y='STATE',
    orientation='h',
    title='Top 20 States by Branch Count',
    labels={'BRANCH_COUNT': 'Number of Branches', 'STATE': 'State'},
    color='BRANCH_COUNT', color_continuous_scale='Greens'
)
fig.update_layout(yaxis=dict(autorange='reversed'), coloraxis_showscale=False, showlegend = False)
fig.show()

---
## 5. ZIP-Level Analysis

In [49]:
#Aggregate branch level data to ZIP level

zip_df = (
    df.groupby("ZIP", as_index=False)
    .agg(
        STATE=('STATE', 'first'),
        TOTAL_DEPOSITS=('DEPOSITS_USD', 'sum'),
        BRANCH_COUNT=('CERT', 'count'),
        LAT=('LAT', 'mean'),
        LON=('LON', 'mean')
    )
)

zip_df['AVG_DEPOSIT_PER_BRANCH'] = zip_df['TOTAL_DEPOSITS'] / zip_df['BRANCH_COUNT']
print(f'Total ZIPs in dataset: {len(zip_df):,}')
zip_df.head(5)

Total ZIPs in dataset: 17,971


,ZIP,STATE,TOTAL_DEPOSITS,BRANCH_COUNT,LAT,LON,AVG_DEPOSIT_PER_BRANCH
0,00601,PR,63000000,1,18.16,-66.72,"63,000,000.00"
1,00602,PR,302762000,2,18.36,-67.17,"151,381,000.00"
2,00603,PR,599215000,4,18.45,-67.13,"149,803,750.00"
3,00604,PR,202000000,1,18.42,-67.15,"202,000,000.00"
4,00606,PR,17000000,1,18.17,-66.94,"17,000,000.00"


In [50]:
# Branches per ZIP distribution

fig = px.histogram(
    zip_df['BRANCH_COUNT'],
    nbins=50,
    title='Distribution of Branch Count per ZIP',
    labels={'value': 'Branches per ZIP'},
    color_discrete_sequence=['#059669'],
)

fig.update_layout(bargap=0.05, showlegend=False)
fig.show()

# Simple counts
print(f'ZIPs with 1 branch  : {(zip_df["BRANCH_COUNT"] == 1).sum():,}')
print(f'ZIPs with 2–5 branches: {zip_df["BRANCH_COUNT"].between(2, 5).sum():,}')
print(f'ZIPs with 6+ branches : {(zip_df["BRANCH_COUNT"] >= 6).sum():,}')

ZIPs with 1 branch  : 6,271
ZIPs with 2–5 branches: 7,196
ZIPs with 6+ branches : 4,504


---
## 6. Pareto Analysis — Deposit Concentration

In [54]:
zip_sorted = zip_df.sort_values('TOTAL_DEPOSITS', ascending = False).reset_index(drop=True)
zip_sorted['CUM_SHARE'] = zip_sorted['TOTAL_DEPOSITS'].cumsum() / zip_sorted['TOTAL_DEPOSITS'].sum()*100
zip_sorted['ZIP_RANK_PCT'] = (zip_sorted.index + 1) / len(zip_sorted) * 100

#Find the 80/20 point
idx_80 = (zip_sorted['CUM_SHARE'] >= 80).idxmax()
pct_zip_for_80_pct = zip_sorted.loc[idx_80, 'ZIP_RANK_PCT']
print(f'Top {pct_zip_for_80_pct:.1f}% of ZIPs hold 80% of total deposits')

Top 16.0% of ZIPs hold 80% of total deposits


In [55]:
fig = go.Figure()
fig.add_trace(go.Scatter(
        x=zip_sorted['ZIP_RANK_PCT'],
        y=zip_sorted['CUM_SHARE'],
        mode ='lines',
        line=dict(color='#2563EB', width=2),
        name='Cumulative deposit share',
))

fig.add_hline(y=80, line_dash='dash', line_color='#EF4444',
              annotation_text='80% of all deposits')

fig.update_layout(title='Pareto: Cumulative Deposit Share by ZIP Percentile',
    xaxis_title='ZIP Code Percentile (sorted by deposits)',
    yaxis_title='Cumulative Share of All Deposits (%)',)
fig.show()

---
## 7. Opportunity Score Analysis

In [56]:
# Normalization
nat_avg_deposits = zip_df['TOTAL_DEPOSITS'].mean()
nat_avg_density = zip_df['BRANCH_COUNT'].mean()

zip_df['NORM_DEPOSIT_BASE'] = zip_df['TOTAL_DEPOSITS'] / nat_avg_deposits
zip_df['NORM_BRANCH_DENSITY'] = zip_df['BRANCH_COUNT'] / nat_avg_density

OPPORTUNITY_WEIGHT_DEPOSIT = 0.6
OPPORTUNITY_WEIGHT_DENSITY = 0.4
CAPTURE_MULTIPLIER = 0.02
CAPTURE_PCT_MIN = 0.01
CAPTURE_PCT_MAX = 0.10

zip_df['OPPORTUNITY_SCORE'] = (OPPORTUNITY_WEIGHT_DEPOSIT*zip_df['NORM_DEPOSIT_BASE'] 
            - OPPORTUNITY_WEIGHT_DENSITY*zip_df['NORM_BRANCH_DENSITY'])
        
zip_df['CAPTURE_PCT'] = (CAPTURE_MULTIPLIER * zip_df['OPPORTUNITY_SCORE']).clip(CAPTURE_PCT_MIN, CAPTURE_PCT_MAX)

zip_df['EXPECTED_CAPTURE_USD'] = zip_df['TOTAL_DEPOSITS'] * zip_df['CAPTURE_PCT']

zip_df = zip_df.sort_values('OPPORTUNITY_SCORE', ascending = False)

zip_df.head(5)

,ZIP,STATE,TOTAL_DEPOSITS,BRANCH_COUNT,LAT,LON,AVG_DEPOSIT_PER_BRANCH,NORM_DEPOSIT_BASE,NORM_BRANCH_DENSITY,OPPORTUNITY_SCORE,CAPTURE_PCT,EXPECTED_CAPTURE_USD
1523,10017,NY,744591256000,20,40.75,-73.98,"37,229,562,800.00",739.16,4.87,441.55,0.10,"74,459,125,600.00"
10571,57108,SD,531388675000,31,43.49,-96.75,"17,141,570,161.29",527.51,7.54,313.49,0.10,"53,138,867,500.00"
15864,84111,UT,485479315000,22,40.76,-111.89,"22,067,241,590.91",481.94,5.35,287.02,0.10,"48,547,931,500.00"
4361,28202,NC,397796853000,10,35.22,-80.84,"39,779,685,300.00",394.89,2.43,235.96,0.10,"39,779,685,300.00"
10568,57105,SD,344742581000,6,43.52,-96.74,"57,457,096,833.33",342.23,1.46,204.75,0.10,"34,474,258,100.00"


In [57]:
fig = px.histogram(
    zip_df,
    x='OPPORTUNITY_SCORE',
    nbins=80,
    title='Distribution of Opportunity Scores Across All ZIPs',
    labels={'OPPORTUNITY_SCORE': 'Opportunity Score'},
    color_discrete_sequence=['#7C3AED'],
)
fig.add_vline(x=0, line_dash='dash', line_color='#EF4444',
              annotation_text='Score = 0')
fig.update_layout(bargap=0.05, showlegend=False)
fig.show()

---
## 8. Top Expansion Opportunities

In [59]:
top20 = zip_df.sort_values('OPPORTUNITY_SCORE', ascending = False).head(20)[[
    'ZIP','STATE', 'TOTAL_DEPOSITS','BRANCH_COUNT','OPPORTUNITY_SCORE','CAPTURE_PCT','EXPECTED_CAPTURE_USD'
]].copy()

#Formatting for readability
def format_currency(x):
    return f"${x:,.0f}"

top20['TOTAL_DEPOSITS']       = top20['TOTAL_DEPOSITS'].apply(format_currency)
top20['EXPECTED_CAPTURE_USD'] = top20['EXPECTED_CAPTURE_USD'].apply(format_currency)
top20['CAPTURE_PCT']          = (top20['CAPTURE_PCT'] * 100).round(1).astype(str) + '%'
top20['OPPORTUNITY_SCORE']    = top20['OPPORTUNITY_SCORE'].round(4)

print('Top 20 ZIPs by Opportunity Score:')
top20

Top 20 ZIPs by Opportunity Score:


,ZIP,STATE,TOTAL_DEPOSITS,BRANCH_COUNT,OPPORTUNITY_SCORE,CAPTURE_PCT,EXPECTED_CAPTURE_USD
1523,10017,NY,"$744,591,256,000",20,441.55,10.0%,"$74,459,125,600"
10571,57108,SD,"$531,388,675,000",31,313.49,10.0%,"$53,138,867,500"
15864,84111,UT,"$485,479,315,000",22,287.02,10.0%,"$48,547,931,500"
4361,28202,NC,"$397,796,853,000",10,235.96,10.0%,"$39,779,685,300"
10568,57105,SD,"$344,742,581,000",6,204.75,10.0%,"$34,474,258,100"
15843,84070,UT,"$327,350,994,000",12,193.81,10.0%,"$32,735,099,400"
3250,19801,DE,"$326,004,592,000",16,192.62,10.0%,"$32,600,459,200"
14546,76262,TX,"$241,408,359,000",11,142.72,10.0%,"$24,140,835,900"
1515,10007,NY,"$230,785,413,000",10,136.49,10.0%,"$23,078,541,300"
1569,10282,NY,"$226,112,000,000",1,134.58,10.0%,"$22,611,200,000"


In [60]:
# Scatter: Deposit base vs Branch density, coloured by Opportunity Score

fig = px.scatter(
    zip_df.head(500),
    x='NORM_BRANCH_DENSITY',
    y='NORM_DEPOSIT_BASE',
    color='OPPORTUNITY_SCORE',
    hover_data=['ZIP','STATE'],
    color_continuous_scale='RdYlGn',
    title= 'Top 500 ZIPs: Normalised Deposit Base vs Branch Density',
    labels={
        'NORM_BRANCH_DENSITY': 'Normalised Branch Density',
        'NORM_DEPOSIT_BASE'  : 'Normalised Deposit Base',
        'OPPORTUNITY_SCORE'  : 'Opp. Score',
    }
)

fig.show()
print('Top-left quadrant = high deposits + low density = best expansion targets.')

Top-left quadrant = high deposits + low density = best expansion targets.


---
## 9. Export Processed Data for Tableau

In [61]:
# ZIP-level metrics — primary Tableau source
zip_out = Output_Dir / 'zip_metrics.csv'
zip_df.to_csv(zip_out, index=False)
print(f'Saved: {zip_out}  ({len(zip_df):,} rows)')

# State-level summary
state_out = Output_Dir / 'state_metrics.csv'
state_df.to_csv(state_out, index=False)
print(f'Saved: {state_out}  ({len(state_df):,} rows)')

# Branch-level (for geographic detail)
branch_cols = ['CERT','BANK_NAME','ADDRESS','BRNUM','UNINUMBR','STATE','STALP','ZIP','LAT','LON','DEPOSITS_USD','YEAR']
branch_out = Output_Dir / 'branch_metrics.csv'
df[branch_cols].to_csv(branch_out, index=False)
print(f'Saved: {branch_out}  ({len(df):,} rows)')

print('\n All outputs ready. Now, we will connect these CSVs in Tableau Desktop.')

Saved: C:\Users\chand\Desktop\Data Science\ORU Course\Bank Branch Expansion Analysis\Dataset\Data\Processed\zip_metrics.csv  (17,971 rows)
Saved: C:\Users\chand\Desktop\Data Science\ORU Course\Bank Branch Expansion Analysis\Dataset\Data\Processed\state_metrics.csv  (58 rows)
Saved: C:\Users\chand\Desktop\Data Science\ORU Course\Bank Branch Expansion Analysis\Dataset\Data\Processed\branch_metrics.csv  (73,876 rows)

 All outputs ready. Now, we will connect these CSVs in Tableau Desktop.
